# GoodForget-RAG Demo

This notebook runs the local TF-IDF toy retrieval experiment. It is a retrieval-control demo, not model unlearning and not a generated-answer evaluation.

In [ ]:
from pathlib import Path

import pandas as pd

from goodforget_rag.data import load_jsonl, vectorizer_training_texts
from goodforget_rag.eval import build_query_result_row, results_dataframe, summarize_results
from goodforget_rag.retrieval import (
    goodforget_retrieve,
    keyword_blocklist_retrieve,
    metadata_filter_retrieve,
    positive_only_retrieve,
    query_rewrite_retrieve,
    vanilla_retrieve,
)
from goodforget_rag.vectorizer import TfidfEncoder

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
documents = load_jsonl(ROOT / "experiments" / "toy_corpus.jsonl")
queries = load_jsonl(ROOT / "experiments" / "toy_queries.jsonl")
encoder = TfidfEncoder().fit(vectorizer_training_texts(documents, queries))
len(documents), len(queries)

In [ ]:
TOP_K = 2
CANDIDATE_K = 5

def run_all(query):
    q = query["query"]
    p = query["positive_intent"]
    f = query["forget_set"]
    return [
        ("Vanilla RAG", vanilla_retrieve(encoder, documents, q, top_k=TOP_K, candidate_k=CANDIDATE_K)),
        ("Positive-only RAG", positive_only_retrieve(encoder, documents, p, top_k=TOP_K, candidate_k=CANDIDATE_K)),
        ("Query Rewrite RAG", query_rewrite_retrieve(encoder, documents, p, top_k=TOP_K, candidate_k=CANDIDATE_K)),
        ("Keyword Blocklist", keyword_blocklist_retrieve(encoder, documents, q, top_k=TOP_K, candidate_k=CANDIDATE_K)),
        ("Metadata Filter", metadata_filter_retrieve(encoder, documents, q, top_k=TOP_K, candidate_k=CANDIDATE_K)),
        ("GoodForget-RAG", goodforget_retrieve(
            encoder,
            documents,
            q,
            p,
            f,
            alpha=0.4,
            beta=0.8,
            gamma=0.2,
            top_k=TOP_K,
            candidate_k=CANDIDATE_K,
        )),
    ]

In [ ]:
rows = []
for query in queries:
    for method, result in run_all(query):
        rows.append(build_query_result_row(method, query, result, documents))

results = results_dataframe(rows)
summary = summarize_results(rows)
summary

## Literal Split Example

Literal forbidden evidence uses obvious words, so keyword blocklists are expected to work relatively well.

In [ ]:
results[results["query_id"] == "q_literal_incident"][["method", "retrieved_doc_ids", "leaked_doc_ids", "utility_recall"]]

## Paraphrase Split Example

Paraphrase cases use code names or indirect descriptions. Keyword blocklists can miss these.

In [ ]:
results[results["query_id"] == "q_paraphrase_cloud"][["method", "retrieved_doc_ids", "leaked_doc_ids", "utility_recall"]]

## Mixed-Evidence Split Example

Mixed documents contain allowed information plus forbidden concepts. Document-level filtering cannot preserve safe spans inside a forbidden document.

In [ ]:
results[results["query_id"] == "q_mixed_ev"][["method", "retrieved_doc_ids", "leaked_doc_ids", "utility_recall"]]

GoodForget-RAG helps when forget-set wording overlaps with forbidden evidence, but it can fail when TF-IDF does not capture indirect semantics or when document-level evidence is mixed. Metadata filtering can outperform it when reliable labels exist.